# Backtesting and Model Validation Dashboard

**Purpose:** Walk-forward VaR backtesting of the AAPL+CAT portfolio, Kupiec unconditional coverage test, Basel traffic-light classification, and a consolidated validation table of all golden-fixture values from the course.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
print('Imports OK')

## Section 1 — Load Data and Construct Portfolio

In [ ]:
from src.schemas import Portfolio, StockPosition

PURCHASE_DATE = '10/13/1997'
SHARES_AAPL   = 24679
SHARES_CAT    = 171
HORIZON       = 5
VAR_CONF      = 0.99
ES_CONF       = 0.975
LOOKBACK      = 2 * 252

aapl = pd.read_csv('../data/AAPL-bloomberg.csv', parse_dates=['Dates'], dayfirst=False)
aapl = aapl.set_index('Dates').sort_index()['PX_LAST']

cat = pd.read_csv('../data/CAT-bloomberg.csv', parse_dates=['Dates'], dayfirst=False)
cat = cat.set_index('Dates').sort_index()['PX_LAST']

prices = pd.DataFrame({'AAPL': aapl, 'CAT': cat}).dropna()
prices = prices[prices.index >= pd.Timestamp(PURCHASE_DATE)]

portfolio_obj = Portfolio(stocks=[
    StockPosition(ticker='AAPL', quantity=SHARES_AAPL),
    StockPosition(ticker='CAT',  quantity=SHARES_CAT),
])

port_val = SHARES_AAPL * prices['AAPL'] + SHARES_CAT * prices['CAT']
pricing_dt = prices.index[-1].date()
print(f'Portfolio value (latest): ${port_val.iloc[-1]:,.2f}')
print(f'Pricing date: {pricing_dt}')

## Section 2 — Current Risk Measures (Parametric and Historical)

In [ ]:
from src.risk.historical import historical_var_es
from src.risk.lognormal import var_long_lognormal, es_long_lognormal

# Historical
hist = historical_var_es(
    portfolio_obj, prices, pricing_dt, LOOKBACK, HORIZON, VAR_CONF, ES_CONF
)

# GBM parametric
port_log_ret = np.log(port_val / port_val.shift(1)).dropna()
w = port_log_ret.tail(LOOKBACK)
mu_d   = w.mean()
sig_d  = w.std()
mu_a   = mu_d * 252 + 0.5 * (sig_d * np.sqrt(252))**2
sig_a  = sig_d * np.sqrt(252)
V0     = port_val.iloc[-1]
gbm_var = var_long_lognormal(V0, mu_a, sig_a, HORIZON / 252, VAR_CONF)
gbm_es  = es_long_lognormal( V0, mu_a, sig_a, HORIZON / 252, ES_CONF)

print(f'{'Method':<30} {'VaR 99%':>15} {'ES 97.5%':>15}')
print('-' * 62)
print(f'{'GBM Parametric':<30} {gbm_var:>15,.0f} {gbm_es:>15,.0f}')
print(f'{'Historical Simulation':<30} {hist["var"]:>15,.0f} {hist["es"]:>15,.0f}')

## Section 3 — Walk-Forward VaR Backtest

In [ ]:
from src.risk.backtest import run_backtest, kupiec_test

print('Running historical walk-forward backtest (may take a moment)...')
# Use last 5 years + 2yr lookback to keep runtime reasonable
prices_bt = prices.tail(7 * 252 + LOOKBACK + HORIZON + 20)

bt_df = run_backtest(
    portfolio=portfolio_obj,
    prices=prices_bt,
    pricing_date=pricing_dt,
    lookback_days=LOOKBACK,
    horizon_days=HORIZON,
    var_confidence=VAR_CONF,
    model='historical',
)

if len(bt_df) == 0:
    reason = bt_df.attrs.get('reason', 'No data')
    print(f'Backtest returned empty: {reason}')
else:
    n_obs   = len(bt_df)
    n_exc   = bt_df['exception'].sum()
    exc_rate = n_exc / n_obs
    print(f'Backtest observations: {n_obs}')
    print(f'VaR exceptions:       {n_exc}')
    print(f'Exception rate:        {exc_rate:.4%}')
    print(f'Expected (at 99%):    {n_obs * 0.01:.1f} exceptions ({0.01:.2%} rate)')

## Section 4 — Kupiec Unconditional Coverage Test

In [ ]:
if len(bt_df) > 0:
    kup = kupiec_test(n_obs, int(n_exc), VAR_CONF)
    print(f'=== Kupiec POF Test ===')
    print(f'  alpha (expected)     = {kup["alpha"]:.4f}')
    print(f'  p_hat (observed)     = {kup["p_hat"]:.4f}')
    print(f'  LR statistic         = {kup["lr_stat"]:.4f}')
    print(f'  p-value              = {kup["p_value"]:.4f}')
    print(f'  Reject H0 at 5%?    = {kup["reject_h0"]}')
else:
    print('Skipping Kupiec test (no backtest data).')
    kup = {'lr_stat': float('nan'), 'p_value': float('nan'), 'reject_h0': False,
           'p_hat': float('nan'), 'alpha': 0.01}

## Section 5 — Basel Traffic-Light Classification

| Zone | Exceptions (out of 250) | Supervisory Response |
|------|------------------------|---------------------|
| Green | 0–4 | No action |
| Yellow | 5–9 | Supervisory scrutiny |
| Red | 10+ | Capital add-on or model rejection |

In [ ]:
def traffic_light(n_exceptions, n_obs=250):
    """Basel market risk traffic light (based on 1-year / 250-day window)."""
    if n_exceptions <= 4:
        return 'GREEN'
    elif n_exceptions <= 9:
        return 'YELLOW'
    else:
        return 'RED'

# Example with standard 250-day window
print(f'Traffic light examples:')
for ex in [0, 2, 4, 5, 7, 9, 10, 15]:
    tl = traffic_light(ex)
    print(f'  {ex:>3} exceptions -> {tl}')

if len(bt_df) > 0:
    # Scale to 250-day equivalent
    exc_250 = int(round(n_exc / n_obs * 250))
    tl_actual = traffic_light(exc_250)
    print(f'\nActual backtest (scaled to 250-day): {exc_250} exceptions -> {tl_actual}')

## Section 6 — Consolidated Golden-Fixture Validation Table

In [ ]:
# Import all needed modules
from src.risk.lognormal import var_long_lognormal, var_short_lognormal
from src.credit.hazard import survival
from src.credit.merton import merton_pd
from src.credit.cds import cds_par_spread_constant_hazard
from src.credit.cva import cva_continuous_constant_exposure
from src.risk.regulatory import risk_weighted_assets

# Compute actuals
golden = [
    ('lognormal', 'var_long_lognormal',
     'V0=10000,mu=0.02,sig=0.2,h=1,p=0.99',
     3720.342,
     var_long_lognormal(10000, 0.02, 0.2, 1, 0.99),
     0.5),
    ('lognormal', 'var_short_lognormal',
     'V0=10000,mu=0.02,sig=0.2,h=1,p=0.99',
     5924.434,
     var_short_lognormal(10000, 0.02, 0.2, 1, 0.99),
     0.5),
    ('hazard', 'survival',
     't=5, lambda=0.0074',
     0.96368,
     survival(5, 0.0074),
     1e-4),
    ('merton', 'merton_pd (Q-measure)',
     'V0=1.1M,B=0.85M,r=0.055,sig=0.28,T=5',
     0.2953,
     merton_pd(1_100_000, 850_000, 0.055, 0.28, 5.0),
     0.001),
    ('cds', 'cds_par_spread_constant_hazard',
     'lambda=0.03, R=0.40',
     0.018,
     cds_par_spread_constant_hazard(0.03, 0.40),
     1e-9),
    ('cva', 'cva_continuous_constant_exposure',
     'K=12,lam=0.03,T=5,R=0.40,r=0',
     1.002903,
     cva_continuous_constant_exposure(12, 0.03, 5, 0.40, r=0.0),
     1e-4),
    ('rwa', 'risk_weighted_assets',
     'assets=[20,50,40],weights=[0,1,0.2]',
     58.0,
     risk_weighted_assets([20.0, 50.0, 40.0], [0.0, 1.0, 0.2]),
     1e-9),
]

print(f'{'Module':<12} {'Function':<35} {'Expected':>12} {'Actual':>12} {'Tol':>8} {'Pass':>6}')
print('-' * 95)
all_pass = True
for module, func, inputs, expected, actual, tol in golden:
    passed = abs(actual - expected) <= tol
    if not passed:
        all_pass = False
    print(f'{module:<12} {func:<35} {expected:>12.6f} {actual:>12.6f} {tol:>8.1e} {str(passed):>6}')

print()
print(f'All goldens pass: {all_pass}')

## Section 7 — VaR Exception Scatter Plot

In [ ]:
if len(bt_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))

    # Realized losses
    ax.plot(bt_df['date'], bt_df['realized_loss'] / 1e6,
            color='steelblue', alpha=0.5, lw=0.7, label='Realized Loss')

    # VaR threshold
    ax.plot(bt_df['date'], bt_df['var_forecast'] / 1e6,
            color='firebrick', lw=1.2, label='VaR Forecast (99%)')

    # Exceptions
    exceptions = bt_df[bt_df['exception'] == 1]
    ax.scatter(exceptions['date'], exceptions['realized_loss'] / 1e6,
               color='red', s=30, zorder=5, label=f'Exceptions (N={len(exceptions)})')

    ax.set_xlabel('Date')
    ax.set_ylabel('Loss ($M)')
    ax.set_title('Walk-Forward VaR Backtest: Realized Losses vs VaR Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    # Synthetic illustration when backtest window is empty
    print('Backtest window was empty; showing synthetic illustration.')
    np.random.seed(42)
    n = 250
    dates_syn = pd.date_range('2023-01-01', periods=n, freq='B')
    losses_syn = np.random.normal(0, 200_000, n)
    var_syn    = np.full(n, 500_000.0)
    exc_mask   = losses_syn > var_syn

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(dates_syn, losses_syn / 1e6, color='steelblue', alpha=0.5, lw=0.7, label='Realized Loss')
    ax.plot(dates_syn, var_syn / 1e6, color='firebrick', lw=1.2, label='VaR Forecast')
    ax.scatter(dates_syn[exc_mask], losses_syn[exc_mask] / 1e6,
               color='red', s=30, zorder=5, label=f'Exceptions (synthetic)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Loss ($M)')
    ax.set_title('Walk-Forward VaR Backtest (Synthetic Illustration)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()